# Extend the training set with inpainted pairs (Colab)

Appends new rows to `combined_training_data_with_dinov2.csv` from an inpainting dataset:

| | |
|---|---|
| `images/` | the original images |
| `outputs/` | the inpainted results |
| `masks/` | the inpaint masks — **not** a feature, kept only as provenance |
| `mapping_merged.xlsx` | ground truth: which output came from which original |

**What gets written**

| transformation | label | pair |
|---|---|---|
| `inpaint` | 1 | `inpaint/images/<orig>` ↔ `inpaint/outputs/<edited>` |
| `inpaint_neg` | 0 | `inpaint/images/<orig>` ↔ `inpaint/outputs/<unrelated>` |

Negatives are generated **1:1** and *diversified*: anchors are drawn round-robin so no single
image dominates. This is the deliberate fix for the flaw in the current data — 1028 of the
1029 `disc21_neg` rows pair against one reference (`R038409`), which lets the model learn
"distance-from-that-one-photo" instead of a general non-match boundary, and inflates the
DINOv2 feature most of all. §7 prints the anchor histogram so you can check it stayed flat.

**Prerequisites**
- Runtime → Change runtime type → **T4 GPU** (DINOv2 on CPU is ~10x slower).
- The Drive folder holding `mapping_merged.xlsx`, `images/`, `outputs/`, `masks/`.
- For the `rhash` recompute in §5, the DISC21 images too (`Dataset/.../queries`, `references`).

---

### ⚠️ Read this before running: the `rhash_dist` column is being rewritten

`RadialHash` changed. The inscribed-radius factor was a hardcoded `0.92`; it is now a
constructor argument, and the value in the working tree is `1.12`.

Verified against the committed images: the existing 2223 rows in
`combined_training_data_with_dinov2.csv` reproduce **exactly** at `0.92` (5/5 sampled rows)
and not at 1.12 or 1.79. The other five hashes reproduce exactly at any setting — `rhash` is
the only column that drifted.

`RHASH_MULTIPLIER` below is set to **1.12** as requested. Because the old rows were computed
at 0.92, this notebook **recomputes `rhash_dist` for every existing row as well**, so the
column means one thing throughout. Appending 1.12 rows next to 0.92 rows without that step
would leave a feature whose value depends on when the row was generated — the model would
happily fit the artefact.

Rows whose images cannot be found are left **blank**, never silently kept at the old value.
The training notebook already drops blank-feature rows in §1.

## 1. Mount Drive, check the GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import torch
print('torch      :', torch.__version__)
print('cuda avail :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device     :', torch.cuda.get_device_name(0))
else:
    print('\nNo GPU. Runtime -> Change runtime type -> T4 GPU, then re-run from cell 1.')

torch      : 2.11.0+cu128
cuda avail : True
device     : Tesla T4


## 2. Get the repo's hash implementations

The six hashers must be **the repo's own classes**, not reimplementations — a subtly
different pHash here would poison the column for every row it touches. Cloning also brings
`hashing/data/own/`, which §5 needs to verify parity against the existing CSV.

Set `REPO_DIR` by hand if you keep a copy of the repo in Drive instead.

In [3]:
import os, subprocess, sys
from pathlib import Path

# ---- Override if you keep the repo in Drive -------------------------------
REPO_DIR = None          # e.g. Path('/content/drive/MyDrive/dam-blockchain')
REPO_URL = 'https://github.com/HammoudYounes/dam-blockchain'
REPO_BRANCH = 'feature/backend-design'
# --------------------------------------------------------------------------

def looks_like_repo(p):
    return p is not None and (Path(p) / 'hashing' / 'algorithms' / 'rhash.py').exists()

if not looks_like_repo(REPO_DIR):
    # A checkout from a previous run in this session?
    for cand in [Path('/content/dam-blockchain'), Path('/content/drive/MyDrive/dam-blockchain')]:
        if looks_like_repo(cand):
            REPO_DIR = cand
            break

if not looks_like_repo(REPO_DIR):
    print(f'cloning {REPO_URL} ({REPO_BRANCH})...')
    r = subprocess.run(
        ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, '/content/dam-blockchain'],
        capture_output=True, text=True)
    print(r.stdout[-2000:], r.stderr[-2000:])
    REPO_DIR = Path('/content/dam-blockchain')

assert looks_like_repo(REPO_DIR), (
    f'No repo at {REPO_DIR}. If it is private, clone it yourself with a token:\n'
    f"  !git clone https://<TOKEN>@github.com/HammoudYounes/dam-blockchain /content/dam-blockchain\n"
    f'or copy the repo into Drive and set REPO_DIR above.')

HASHING_DIR = Path(REPO_DIR) / 'hashing'
OWN_ROOT = HASHING_DIR / 'data' / 'own'         # holds images/ and images_variants/
if str(HASHING_DIR) not in sys.path:
    sys.path.insert(0, str(HASHING_DIR))

print('REPO_DIR   :', REPO_DIR)
print('OWN_ROOT   :', OWN_ROOT, '(exists:', OWN_ROOT.exists(), ')')

cloning https://github.com/HammoudYounes/dam-blockchain (feature/backend-design)...
 Cloning into '/content/dam-blockchain'...

REPO_DIR   : /content/dam-blockchain
OWN_ROOT   : /content/dam-blockchain/hashing/data/own (exists: True )


In [ ]:
from algorithms.ahash import AverageHash
from algorithms.phash import PerceptualHash
from algorithms.dhash import DifferenceHash
from algorithms.HSVHash import HSVColorHash
from algorithms.rhash import RadialHash
from algorithms.Chash import ColorHash

# The value this whole run uses. See the warning at the top: the existing rows were
# generated at 0.92, so §5 recomputes them at this value rather than mixing the two.
RHASH_MULTIPLIER = 1.12

def build_hashers():
    # Fresh hasher instances. rhash is the only one that takes a parameter.
    return {
        'ahash':   AverageHash(),
        'phash':   PerceptualHash(),
        'dhash':   DifferenceHash(),
        'hsvhash': HSVColorHash(),
        'rhash':   RadialHash(multiplier=RHASH_MULTIPLIER),
        'chash':   ColorHash(),
    }

HASHERS = build_hashers()
HASH_COLS = [f'{k}_dist' for k in HASHERS]
FEATURE_COL = 'dinov2_dist'

print('RHASH_MULTIPLIER =', RHASH_MULTIPLIER)
for k, h in HASHERS.items():
    print(f'  {k:<8} HASH_BITS={h.HASH_BITS}')

TypeError: RadialHash() takes no arguments

## 3. Locate the inputs

One bounded walk of Drive. If it picks the wrong candidate, set the paths by hand at the top
and re-run — Drive's FUSE mount is slow enough that a full scan is not worth it.

In [ ]:
# ---- Set manually to override auto-discovery -----------------------------
INPAINT_ROOT = None   # dir holding images/, outputs/, masks/, mapping_merged.xlsx
XLSX_PATH    = None   # mapping_merged.xlsx
INPUT_CSV    = None   # combined_training_data_with_dinov2.csv
DISC21_ROOT  = None   # dir holding queries/ and references/  (needed for the rhash recompute)
OUT_DIR      = None   # defaults to INPUT_CSV's folder
# --------------------------------------------------------------------------

DRIVE = Path('/content/drive/MyDrive')
MAX_DEPTH = 7
CSV_NAME = 'combined_training_data_with_dinov2.csv'
XLSX_NAME = 'mapping_merged.xlsx'

def scan(root, max_depth=MAX_DEPTH):
    inpaint, disc21, csvs, xlsxs = [], [], [], []
    root = Path(root)
    for dirpath, dirnames, filenames in os.walk(root):
        p = Path(dirpath)
        if len(p.relative_to(root).parts) >= max_depth:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith('.')]
        if (p / 'images').is_dir() and (p / 'outputs').is_dir():
            inpaint.append(p)
        if (p / 'queries').is_dir() and (p / 'references').is_dir():
            disc21.append(p)
        if CSV_NAME in filenames:
            csvs.append(p / CSV_NAME)
        if XLSX_NAME in filenames:
            xlsxs.append(p / XLSX_NAME)
    return inpaint, disc21, csvs, xlsxs

if any(v is None for v in (INPAINT_ROOT, XLSX_PATH, INPUT_CSV, DISC21_ROOT)):
    print('Scanning Drive (takes a minute)...')
    i_hits, d_hits, c_hits, x_hits = scan(DRIVE)
    for label, hits in [('inpaint roots', i_hits), ('disc21 roots', d_hits),
                        (CSV_NAME, c_hits), (XLSX_NAME, x_hits)]:
        print(f'\n{label}: {len(hits)} hit(s)')
        for h in hits[:5]:
            print('   ', h)
    INPAINT_ROOT = INPAINT_ROOT or (i_hits[0] if i_hits else None)
    DISC21_ROOT  = DISC21_ROOT  or (d_hits[0] if d_hits else None)
    INPUT_CSV    = INPUT_CSV    or (c_hits[0] if c_hits else None)
    XLSX_PATH    = XLSX_PATH    or (x_hits[0] if x_hits else None)

assert INPAINT_ROOT, 'No folder with both images/ and outputs/ found — set INPAINT_ROOT.'
assert XLSX_PATH, f'{XLSX_NAME} not found — set XLSX_PATH.'
assert INPUT_CSV, f'{CSV_NAME} not found — set INPUT_CSV.'

INPAINT_ROOT = Path(INPAINT_ROOT)
IMAGES_DIR  = INPAINT_ROOT / 'images'
OUTPUTS_DIR = INPAINT_ROOT / 'outputs'
MASKS_DIR   = INPAINT_ROOT / 'masks'
OUT_DIR = Path(OUT_DIR) if OUT_DIR else Path(INPUT_CSV).parent

print('\n--- resolved ---')
print('INPAINT_ROOT :', INPAINT_ROOT)
print('XLSX_PATH    :', XLSX_PATH)
print('INPUT_CSV    :', INPUT_CSV)
print('DISC21_ROOT  :', DISC21_ROOT, '(needed only for the rhash recompute)')
print('OUT_DIR      :', OUT_DIR)

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}

def inventory(d):
    # {filename: path} and {stem: path} for one folder, images only.
    by_name, by_stem = {}, {}
    if not Path(d).is_dir():
        return by_name, by_stem
    for p in sorted(Path(d).iterdir()):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            by_name[p.name] = p
            by_stem.setdefault(p.stem, p)
    return by_name, by_stem

IMG_BY_NAME,  IMG_BY_STEM  = inventory(IMAGES_DIR)
OUT_BY_NAME,  OUT_BY_STEM  = inventory(OUTPUTS_DIR)
MASK_BY_NAME, MASK_BY_STEM = inventory(MASKS_DIR)

print(f'images/  : {len(IMG_BY_NAME)} files')
print(f'outputs/ : {len(OUT_BY_NAME)} files')
print(f'masks/   : {len(MASK_BY_NAME)} files')
for label, d in [('images', IMG_BY_NAME), ('outputs', OUT_BY_NAME)]:
    print(f'\nsample {label}/:', [n for n in list(d)[:5]])

## 4. Read `mapping_merged.xlsx` and find the pair columns

Rather than hardcoding column names, each column is scored by **how many of its values
actually resolve to a file** in `images/` and in `outputs/`. The best-matching column for
each side wins. That survives renames, extra columns, full-vs-bare paths, and missing
extensions.

Check the printed table before continuing. To override, set `COL_A` / `COL_B` in the next cell.

In [ ]:
import pandas as pd

SHEET = 0            # sheet name or index
xl = pd.ExcelFile(XLSX_PATH)
print('sheets:', xl.sheet_names)

mdf = pd.read_excel(XLSX_PATH, sheet_name=SHEET)
print(f'\nrows: {len(mdf)}   columns: {list(mdf.columns)}')
print('\ndtypes:'); print(mdf.dtypes.to_string())
print('\nhead:')
display(mdf.head(8))

In [ ]:
def basename(v):
    # xlsx cells may hold a bare name, a name without extension, or a full path.
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    s = str(v).strip().replace('\\', '/')
    return s.rsplit('/', 1)[-1] if s else None

def score_column(series, by_name, by_stem):
    # Fraction of non-null values in this column that resolve to a real file.
    vals = [basename(v) for v in series]
    vals = [v for v in vals if v]
    if not vals:
        return 0.0, 0
    hits = sum(1 for v in vals if v in by_name or Path(v).stem in by_stem)
    return hits / len(vals), hits

score_rows = []
for c in mdf.columns:
    ia, ha = score_column(mdf[c], IMG_BY_NAME, IMG_BY_STEM)
    ob, hb = score_column(mdf[c], OUT_BY_NAME, OUT_BY_STEM)
    mk, hm = score_column(mdf[c], MASK_BY_NAME, MASK_BY_STEM)
    score_rows.append({'column': c, 'resolves in images/': ia, 'resolves in outputs/': ob,
                       'resolves in masks/': mk, 'n_hits_images': ha, 'n_hits_outputs': hb})
score_df = pd.DataFrame(score_rows)
display(score_df)

# ---- Override the auto-pick here if the table says otherwise --------------
COL_A = None      # column naming the ORIGINAL image  (-> images/)
COL_B = None      # column naming the INPAINTED image (-> outputs/)
COL_TRANSFORM = None   # optional: a column describing the edit; becomes inpaint_<value>
# --------------------------------------------------------------------------

MIN_COVERAGE = 0.5
if COL_A is None:
    best = score_df.loc[score_df['resolves in images/'].idxmax()]
    COL_A = best['column'] if best['resolves in images/'] >= MIN_COVERAGE else None
if COL_B is None:
    best = score_df.loc[score_df['resolves in outputs/'].idxmax()]
    COL_B = best['column'] if best['resolves in outputs/'] >= MIN_COVERAGE else None

assert COL_A is not None, (
    'No column resolves to >=50% of images/. Set COL_A by hand, or check that '
    'IMAGES_DIR points at the right folder.')
assert COL_B is not None, (
    'No column resolves to >=50% of outputs/. Set COL_B by hand.')
assert COL_A != COL_B, f'COL_A and COL_B both picked {COL_A!r} — set them by hand.'

print(f'COL_A (original) = {COL_A!r}')
print(f'COL_B (inpainted) = {COL_B!r}')
print(f'COL_TRANSFORM     = {COL_TRANSFORM!r}')

## 5. Parity gate — recompute `rhash_dist` for the existing rows

Two jobs, in one place, before anything new is generated:

1. **Verify** the five stable hashes still reproduce the existing CSV bit-for-bit on rows
   whose images are in the repo. If they don't, the environment differs from the one that
   built the CSV and nothing below should be trusted — the assert stops the run.
2. **Recompute** `rhash_dist` at `RHASH_MULTIPLIER` for every existing row that can be
   resolved, so old and new rows share one definition.

Rows whose images are unavailable get a blank `rhash_dist`, not a stale one.

In [ ]:
import csv as _csv

with open(INPUT_CSV, newline='') as f:
    reader = _csv.DictReader(f)
    fieldnames = list(reader.fieldnames or [])
    existing_rows = list(reader)

print(f'existing rows: {len(existing_rows)}')
print('columns      :', fieldnames)

OWN_PREFIXES    = ('images/', 'images_variants/')
DISC21_PREFIXES = ('references/', 'queries/')
INPAINT_PREFIX  = 'inpaint/'          # new rows; distinct so it cannot collide with images/

def resolve_existing(rel):
    # Map a CSV path onto disk. Mirrors add_dinov2_distance.py dispatch.
    rel = str(rel).strip().replace('\\', '/')
    if rel.startswith(INPAINT_PREFIX):
        sub = rel[len(INPAINT_PREFIX):]
        if sub.startswith('images/'):
            return IMAGES_DIR / sub[len('images/'):]
        if sub.startswith('outputs/'):
            return OUTPUTS_DIR / sub[len('outputs/'):]
        return None
    if rel.startswith(OWN_PREFIXES):
        return OWN_ROOT / rel
    if rel.startswith(DISC21_PREFIXES):
        return (Path(DISC21_ROOT) / rel) if DISC21_ROOT else None
    return None

_hash_cache = {name: {} for name in HASHERS}

def hash_of(algo, path):
    c = _hash_cache[algo]
    k = str(path)
    if k not in c:
        c[k] = HASHERS[algo].compute(k)
    return c[k]

def distances(path_a, path_b, algos=None):
    out = {}
    for algo in (algos or HASHERS):
        h = HASHERS[algo]
        out[f'{algo}_dist'] = h.hamming_distance(hash_of(algo, path_a),
                                                 hash_of(algo, path_b)) / h.HASH_BITS
    return out

In [ ]:
# ---- 1. Verify the five stable hashes reproduce the CSV -------------------
STABLE = ['ahash', 'phash', 'dhash', 'hsvhash', 'chash']
CHECK_N = 8

checked, mismatches = 0, []
for r in existing_rows:
    if checked >= CHECK_N:
        break
    pa, pb = resolve_existing(r['image_a']), resolve_existing(r['image_b'])
    if not (pa and pb and pa.exists() and pb.exists()):
        continue
    got = distances(pa, pb, STABLE)
    for algo in STABLE:
        col = f'{algo}_dist'
        want = r.get(col, '')
        if want in ('', None):
            continue
        if abs(got[col] - float(want)) > 1e-9:
            mismatches.append((r['image_b'], col, float(want), got[col]))
    checked += 1

print(f'verified {checked} row(s) against the CSV')
if mismatches:
    print('\nMISMATCHES:')
    for name, col, want, got in mismatches[:20]:
        print(f'  {name:<45} {col:<14} csv={want:.6f}  recomputed={got:.6f}')
assert checked > 0, (
    'Could not resolve any existing row to disk — check REPO_DIR / DISC21_ROOT. '
    'Without this check the recompute below is unverified.')
assert not mismatches, (
    'The five stable hashes do not reproduce the existing CSV. The hashing code or the '
    'images differ from those that built it; fix that before appending anything.')
print('OK — the 5 stable hashes reproduce the CSV exactly. Only rhash will be rewritten.')

In [ ]:
# ---- 2. Recompute rhash_dist for every existing row ----------------------
rh = HASHERS['rhash']
changed, unchanged, blanked = 0, 0, 0

for i, r in enumerate(existing_rows, 1):
    pa, pb = resolve_existing(r['image_a']), resolve_existing(r['image_b'])
    if not (pa and pb and pa.exists() and pb.exists()):
        r['rhash_dist'] = ''        # blank, never a stale 0.92 value
        blanked += 1
        continue
    new = rh.hamming_distance(hash_of('rhash', pa), hash_of('rhash', pb)) / rh.HASH_BITS
    old = r.get('rhash_dist', '')
    try:
        if abs(float(old) - new) > 1e-9:
            changed += 1
        else:
            unchanged += 1
    except (TypeError, ValueError):
        changed += 1
    r['rhash_dist'] = f'{new:.17g}'
    if i % 250 == 0 or i == len(existing_rows):
        print(f'  {i}/{len(existing_rows)} rows')

print(f'\nrhash_dist @ multiplier {RHASH_MULTIPLIER}:')
print(f'  changed   : {changed}')
print(f'  unchanged : {unchanged}')
print(f'  blanked   : {blanked}   (images not on disk)')
if blanked:
    print('\n  Blank rows will be DROPPED by the training notebook (§1). If that count is\n'
          '  large, point DISC21_ROOT at the DISC21 images and re-run this cell.')

## 6. Build the positive pairs

One row per `(original, inpainted)` entry in the mapping. Paths are written with an
`inpaint/` prefix — deliberately **not** `images/`, which already means
`hashing/data/own/images/` for the existing rows. Reusing it would make the two datasets
indistinguishable to every downstream resolver.

In [ ]:
def resolve_side(name, by_name, by_stem):
    b = basename(name)
    if not b:
        return None
    if b in by_name:
        return by_name[b]
    return by_stem.get(Path(b).stem)

positives, unresolved = [], []
seen_pairs = set()

for _, row in mdf.iterrows():
    pa = resolve_side(row[COL_A], IMG_BY_NAME, IMG_BY_STEM)
    pb = resolve_side(row[COL_B], OUT_BY_NAME, OUT_BY_STEM)
    if pa is None or pb is None:
        unresolved.append((row[COL_A], row[COL_B]))
        continue
    rel_a = f'inpaint/images/{pa.name}'
    rel_b = f'inpaint/outputs/{pb.name}'
    if (rel_a, rel_b) in seen_pairs:
        continue
    seen_pairs.add((rel_a, rel_b))
    tf = 'inpaint'
    if COL_TRANSFORM:
        v = row.get(COL_TRANSFORM)
        if v is not None and not (isinstance(v, float) and pd.isna(v)):
            tf = f'inpaint_{str(v).strip().replace(" ", "_")}'
    positives.append({'image_a': rel_a, 'image_b': rel_b,
                      'transformation': tf, 'label': 1,
                      'path_a': pa, 'path_b': pb})

print(f'mapping rows      : {len(mdf)}')
print(f'positive pairs    : {len(positives)}')
print(f'unresolved        : {len(unresolved)}')
if unresolved:
    print('\nfirst few unresolved (name not found in images/ or outputs/):')
    for a, b in unresolved[:8]:
        print(f'  {a!r}  ->  {b!r}')
assert positives, 'No positive pairs resolved — check COL_A/COL_B and the folder paths.'

# 1:N is fine (several edits per original); report it so the number is not a surprise.
from collections import Counter
per_src = Counter(p['image_a'] for p in positives)
print(f'\ndistinct originals used : {len(per_src)}')
print(f'edits per original      : min {min(per_src.values())}, '
      f'max {max(per_src.values())}, mean {sum(per_src.values())/len(per_src):.2f}')
print('transformation classes  :', dict(Counter(p["transformation"] for p in positives)))

## 7. Diversified negatives, 1:1

Each negative pairs an original with an inpainted output that did **not** come from it.
Anchors are walked round-robin over the shuffled list of originals, so every original
contributes about the same number of negatives.

The printed histogram is the point of this section. For reference, the existing
`disc21_neg` pool has a top-anchor share of **99.9%** (1028 / 1029). Anything above a few
percent here means the loop failed and the new rows carry the same defect.

In [ ]:
import random
from itertools import count

NEG_RATIO = 1
RANDOM_SEED = 42

rng = random.Random(RANDOM_SEED)

gt_pairs = {(p['image_a'], p['image_b']) for p in positives}
# Which original produced each output — a negative must not reuse its own source, even
# if the mapping lists that pair under a different row.
src_of_output = {p['image_b']: p['image_a'] for p in positives}

originals = sorted({p['image_a'] for p in positives})
outputs = sorted({p['image_b'] for p in positives})
path_of = {p['image_a']: p['path_a'] for p in positives}
path_of.update({p['image_b']: p['path_b'] for p in positives})

n_neg_target = len(positives) * NEG_RATIO
anchors = list(originals)
rng.shuffle(anchors)

negatives = []
used = set()
exhausted = 0

for i in count():
    if len(negatives) >= n_neg_target:
        break
    if exhausted >= len(anchors):        # every anchor is out of valid partners
        break
    a = anchors[i % len(anchors)]
    if i % len(anchors) == 0 and i:
        rng.shuffle(outputs)
    choices = [b for b in outputs
               if src_of_output.get(b) != a
               and (a, b) not in gt_pairs
               and (a, b) not in used]
    if not choices:
        exhausted += 1
        continue
    exhausted = 0
    b = rng.choice(choices)
    used.add((a, b))
    negatives.append({'image_a': a, 'image_b': b,
                      'transformation': 'inpaint_neg', 'label': 0,
                      'path_a': path_of[a], 'path_b': path_of[b]})

print(f'negatives requested : {n_neg_target}')
print(f'negatives generated : {len(negatives)}')
if len(negatives) < n_neg_target:
    print('  (fewer than requested — not enough distinct non-matching combinations)')

import math

anchor_counts = Counter(n['image_a'] for n in negatives)
max_count = max(anchor_counts.values())
top_share = max_count / len(negatives) if negatives else 0

# Scale-free gate. Perfect round-robin gives every anchor floor or ceil of
# n_neg/n_anchors, so that ceiling (plus a little slack) is the bar. A fixed percentage
# would fail spuriously on a small dataset and pass a concentrated large one.
fair_max = math.ceil(len(negatives) / max(len(anchor_counts), 1)) + 1

print(f'\ndistinct anchors    : {len(anchor_counts)}')
print(f'per-anchor negatives: min {min(anchor_counts.values())}, max {max_count}  '
      f'(round-robin bound: {fair_max})')
print(f'top-anchor share    : {top_share:.2%}   (disc21_neg, for comparison: 99.90%)')
assert max_count <= fair_max, (
    f'One anchor owns {max_count} of {len(negatives)} negatives ({top_share:.1%}), above the '
    f'round-robin bound of {fair_max} — that reintroduces exactly the leak this section '
    f'exists to avoid. Check the loop.')
print('\nOK — negatives are spread evenly across anchors.')

## 8. Hash distances for the new pairs

In [ ]:
new_pairs = positives + negatives
print(f'new pairs: {len(new_pairs)}  ({len(positives)} pos / {len(negatives)} neg)')

new_rows = []
for i, p in enumerate(new_pairs, 1):
    feats = distances(p['path_a'], p['path_b'])
    new_rows.append({'image_a': p['image_a'], 'image_b': p['image_b'],
                     'transformation': p['transformation'], 'label': p['label'],
                     **{k: f'{v:.17g}' for k, v in feats.items()}})
    if i % 100 == 0 or i == len(new_pairs):
        print(f'  {i}/{len(new_pairs)} pairs hashed')

print('\nMean hash distance by class (new rows only):')
print(f"{'feature':<14}{'pos':>10}{'neg':>10}{'gap':>10}")
print('-' * 44)
for col in HASH_COLS:
    pv = [float(r[col]) for r in new_rows if r['label'] == 1]
    nv = [float(r[col]) for r in new_rows if r['label'] == 0]
    print(f'{col:<14}{sum(pv)/len(pv):>10.4f}{sum(nv)/len(nv):>10.4f}'
          f'{sum(nv)/len(nv) - sum(pv)/len(pv):>10.4f}')
print('\ngap = negative mean - positive mean. Near zero means the hash cannot see this edit.')

## 9. `dinov2_dist` for the new pairs

Same arithmetic as `add_dinov2_distance.py`: open → RGB → resize 224 → processor → CLS token
→ L2 normalise, then `1 - dot(va, vb)`. `MODEL_SIZE` must match the retriever's `MODEL_SIZE`
(`small` by default in `hashing/main.py`) or the training feature and the served feature come
from different models.

Embeddings are checkpointed to Drive; re-running after a disconnect resumes from the cache.

In [ ]:
from transformers import AutoImageProcessor, AutoModel
import numpy as np

MODEL_SIZE = 'small'          # small | base | large | giant — must match the retriever

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
name = f'facebook/dinov2-{MODEL_SIZE}'
print('loading', name, '->', device)
proc = AutoImageProcessor.from_pretrained(name)
dino = AutoModel.from_pretrained(name).eval().to(device)
print('loaded')

In [ ]:
from PIL import Image
from tqdm.auto import tqdm

BATCH = 32
CACHE_PATH = OUT_DIR / f'dinov2_{MODEL_SIZE}_inpaint_embeddings.npz'

emb = {}
if CACHE_PATH.exists():
    z = np.load(CACHE_PATH, allow_pickle=False)
    emb = {k: v for k, v in zip(z['keys'].tolist(), z['vecs'])}
    print(f'resumed {len(emb)} embeddings from {CACHE_PATH}')

def save_emb():
    if emb:
        np.savez(CACHE_PATH, keys=np.array(list(emb.keys())), vecs=np.stack(list(emb.values())))

def embed(paths):
    imgs, kept = [], []
    for sp in paths:
        try:
            imgs.append(Image.open(sp).convert('RGB').resize((224, 224)))
            kept.append(sp)
        except Exception as e:
            print('skip', sp, e)
    if not imgs:
        return
    inputs = proc(images=imgs, return_tensors='pt')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        h = dino(**inputs).last_hidden_state[:, 0, :]
        h = h / h.norm(dim=-1, keepdim=True)
    for sp, v in zip(kept, h.cpu().numpy()):
        emb[sp] = v

# One embedding per unique image, not per pair — outputs are reused across negatives.
unique_paths = sorted({str(p['path_a']) for p in new_pairs} | {str(p['path_b']) for p in new_pairs})
todo = [p for p in unique_paths if p not in emb]
print(f'unique images: {len(unique_paths)}   to embed: {len(todo)}')

for i in tqdm(range(0, len(todo), BATCH)):
    embed(todo[i:i + BATCH])
    if (i // BATCH) % 20 == 0:
        save_emb()
save_emb()
print('cached embeddings:', len(emb))

In [ ]:
blank = 0
for r, p in zip(new_rows, new_pairs):
    va, vb = emb.get(str(p['path_a'])), emb.get(str(p['path_b']))
    if va is None or vb is None:
        r[FEATURE_COL] = ''            # blank, never silently zero
        blank += 1
        continue
    cos = float(np.clip(np.dot(va, vb), -1.0, 1.0))
    r[FEATURE_COL] = f'{1.0 - cos:.6f}'

print(f'filled {len(new_rows) - blank}/{len(new_rows)} rows')
if blank:
    print(f'{blank} row(s) left BLANK — the training notebook drops these')

pv = [float(r[FEATURE_COL]) for r in new_rows if r['label'] == 1 and r[FEATURE_COL] != '']
nv = [float(r[FEATURE_COL]) for r in new_rows if r['label'] == 0 and r[FEATURE_COL] != '']
if pv and nv:
    print(f'\n{FEATURE_COL}: mean(copy) = {sum(pv)/len(pv):.4f}   '
          f'mean(different) = {sum(nv)/len(nv):.4f}   gap = {sum(nv)/len(nv) - sum(pv)/len(pv):.4f}')

## 10. Append and save

Written to a **new filename**, leaving the input CSV untouched — the recompute in §5 rewrote
a column, and overwriting the only copy of a dataset in place is how you lose the ability to
tell what a model was trained on. The old file stays valid as the 0.92-rhash snapshot.

In [ ]:
OUTPUT_CSV = OUT_DIR / 'combined_training_data_with_dinov2_inpaint.csv'

for c in HASH_COLS + [FEATURE_COL]:
    if c not in fieldnames:
        fieldnames.append(c)

all_rows = existing_rows + new_rows
with open(OUTPUT_CSV, 'w', newline='') as f:
    w = _csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    w.writeheader()
    w.writerows(all_rows)

print(f'saved {len(all_rows)} rows -> {OUTPUT_CSV}')
print(f'  existing : {len(existing_rows)}')
print(f'  new      : {len(new_rows)}')

def lab(rs, v):
    return sum(1 for r in rs if str(r.get('label', '')).strip() == str(v))

print(f'\nlabel balance: {lab(all_rows, 1)} positives / {lab(all_rows, 0)} negatives '
      f'(positive rate = {lab(all_rows, 1)/len(all_rows):.2%})')
print(f'  was        : {lab(existing_rows, 1)} / {lab(existing_rows, 0)} '
      f'({lab(existing_rows, 1)/len(existing_rows):.2%})')

print('\nrows per transformation:')
for tf, n in sorted(Counter(r['transformation'] for r in all_rows).items(),
                    key=lambda kv: -kv[1]):
    print(f'  {tf:<20} {n}')

## 11. Sanity checks on the combined file

Per-class separation for each feature, split by subset. A gap that looks healthy overall can
hide a subset sitting at chance — which is exactly what the six hashes do on DISC21.

In [ ]:
def fcol(rs, name):
    out = []
    for r in rs:
        v = r.get(name, '')
        if v not in ('', None):
            try:
                out.append(float(v))
            except ValueError:
                pass
    return np.array(out)

def table(rs, title):
    pos = [r for r in rs if str(r.get('label', '')).strip() == '1']
    neg = [r for r in rs if str(r.get('label', '')).strip() == '0']
    if not pos or not neg:
        print(f'\n{title}: only one class present, skipped')
        return
    print(f'\n{title}  ({len(pos)} pos / {len(neg)} neg)')
    print(f"{'feature':<14}{'pos':>10}{'neg':>10}{'gap':>10}")
    print('-' * 44)
    for nm in HASH_COLS + [FEATURE_COL]:
        p, n = fcol(pos, nm), fcol(neg, nm)
        if p.size == 0 or n.size == 0:
            continue
        print(f'{nm:<14}{p.mean():>10.4f}{n.mean():>10.4f}{n.mean() - p.mean():>10.4f}')

def tf_of(r):
    return str(r.get('transformation', ''))

table(all_rows, 'ALL ROWS')
table([r for r in all_rows if tf_of(r).startswith('inpaint')], 'INPAINT ONLY (new)')
table([r for r in all_rows if tf_of(r).startswith('disc21')], 'DISC21 ONLY')
table([r for r in all_rows if not tf_of(r).startswith(('inpaint', 'disc21'))], 'OWN DATASET ONLY')
print('\ngap = negative mean - positive mean. Larger is better separation.')

In [ ]:
# Negative-anchor concentration across the whole file. This is the number that made the
# old headline metrics optimistic; keep an eye on it as the dataset grows.
print(f"{'transformation':<20}{'neg rows':>10}{'anchors':>9}{'top share':>11}")
print('-' * 50)
for tf in sorted({tf_of(r) for r in all_rows if str(r.get('label', '')).strip() == '0'}):
    rs = [r for r in all_rows if tf_of(r) == tf and str(r.get('label', '')).strip() == '0']
    c = Counter(r['image_a'] for r in rs)
    print(f'{tf:<20}{len(rs):>10}{len(c):>9}{max(c.values())/len(rs):>10.1%}')
print('\nA top share near 100% means that subset teaches "not this one image" rather than\n'
      'a general non-match boundary.')

## What to change in the training notebook

`copymint_logreg_baseline_ver1.ipynb` will not pick these rows up correctly as-is:

1. **Point it at the new file** — `CSV_NAME = 'combined_training_data_with_dinov2_inpaint.csv'`.
2. **The hard-subset masks are `disc21`-only.** Two places assume it:
   - §1 `disc_mask = df['transformation'].str.startswith('disc21')`
   - §4 `INPAINTING_TFS = ['disc21', 'disc21_neg']`, which feeds `inpaint_mask` and therefore
     every `*_disc21` column in the §4b ablation.

   Both need to include the `inpaint*` classes, or the new rows silently land in the
   "classical augmentation" bucket and the ablation's hard-subset numbers stay unchanged
   while the headline numbers move. Better: split them into three explicit buckets
   (`own` / `disc21` / `inpaint`) and report all three — the whole reason for adding
   inpainted pairs is to see them separately.
3. **Re-tune the threshold.** §7's operating point belongs to the old model and the old
   `rhash` definition.
4. **Bump the model version** when you serialize. `verifications.model_version` in the
   backend exists so a past verdict can be traced to the model that produced it; reusing
   `7feat-dinov2` after a data and feature change breaks that.

Say the word and I'll make edits 1–3 in the training notebook.